# DeepEval Multi-Turn Evaluation Metrics

## Overview

This notebook demonstrates how to evaluate multi-turn conversations using the [DeepEval evaluation framework](https://deepeval.com/guides/guides-ai-agent-evaluation). DeepEval generates synthetic conversations from predefined scenarios and scores GenAI application behavior across multiple quality dimensions, from individual response coherence to full-session goal achievement.

Unlike single-turn evaluation, where each query stands alone, multi-turn conversations are interdependent: every turn builds on the full history before it, so one weak response can derail the rest of the session. This means you cannot predefine "correct" exchanges several turns deep, and quality must be measured at the session level, not per turn. DeepEval addresses this by evaluating against scenarios rather than fixed input-output pairs.

## What You'll Learn

1. How to use `ConversationalGolden` scenarios to define evaluation use cases and determine which types of use cases to cover.
2. How to use `ConversationSimulator` to generate synthetic multi-turn conversations, simulate them with your GenAI application, and collect results as `ConversationalTestCase` objects.
3. Which built-in metrics to use for different types of multi-turn evaluations, including conversation quality, behavioral compliance, and agentic metrics.
4. How to apply these metrics and interpret their scores to identify specific areas of weakness.
5. How to define custom evaluations tailored to your use cases using `ConversationalGEval` and `ConversationalDAGMetric`.

## Key Concepts

DeepEval's multi-turn evaluation flow has three core components:

- **ConversationalGolden** — A test specification that defines *what* to evaluate: a scenario description, a user persona, and an expected outcome. No scripted messages, just constraints.
- **ConversationSimulator** — Generates realistic user messages turn by turn, sends each to your application, and continues until the expected outcome is reached or a turn limit is hit.
- **ConversationalTestCase** — The resulting conversation (all user and assistant turns in order), packaged for scoring against multi-turn metrics.

Metrics are grouped into four categories: *Conversation Quality* (completeness, relevance, knowledge retention), *Behavioral Compliance* (role and topic adherence), *Agentic* (goal accuracy, tool use), and *Custom* (developer-defined criteria via `ConversationalGEval` and `ConversationalDAGMetric`).

# Setup

For this guide, we will use a GenAI travel booking assistant built with Strands Agents framework presented in the previous notebooks. The assistant handles the end-to-end workflow of planning and managing travel: searching for flights and hotels, placing bookings, retrieving existing reservations, and processing cancellations.

This application makes a strong evaluation target - it naturally demands multi-turn interaction. A user rarely searches, selects, and books in a single message. Instead, a typical session might span half a dozen turns: asking for flights, narrowing options by price, switching to hotel search, booking both, and then circling back to modify a reservation. That conversational depth is exactly what makes multi-turn evaluation essential.

The architecture is intentionally simple. At its core, it is an LLM-powered conversational agent equipped with different tools that give it the ability to act on user requests. Booking state is stored in a simple in-memory dictionary. 

For simlicity and consistency of demonstration, search results are drawn from hardcoded mock data. This keeps the agent self-contained and deterministic on the tool side, ensuring that any variability in evaluation results stems from the LLM's conversational behavior, not from external dependencies.

We start from installing the requirements for this project.

In [ ]:
# Install all required packages (deepeval, strands, boto3, etc.)
%pip install -r requirements.txt

The following code imports the required agent and tool components from Strands framework.

In [ ]:
# Import Strands Agent class and tool decorator for defining agent tools
from strands import Agent, tool
from datetime import date

Now, let's define the assistant's code. The cell below defines the in-memory booking state and six tools that the agent will use: `search_flights`, `search_hotels`, `book_flight`, `book_hotel`, `get_bookings`, and `cancel_booking`. Each tool returns a formatted string that the agent relays to the user.

In [ ]:
# --- In-memory booking state ---
bookings: dict = {}
booking_counter = 0


# --- Tool definitions: each @tool becomes available to the agent ---

@tool
def search_flights(origin: str, destination: str, departure_date: str, return_date: str = None) -> str:
    """Search for available flights between two cities.

    Args:
        origin: Departure city or airport code (e.g. 'New York' or 'JFK')
        destination: Arrival city or airport code (e.g. 'London' or 'LHR')
        departure_date: Departure date in YYYY-MM-DD format
        return_date: Optional return date in YYYY-MM-DD format for round trips
    """
    flights = [
        {"flight": "AA101", "airline": "American Airlines", "departs": "08:00", "arrives": "20:00", "price": 450, "class": "Economy"},
        {"flight": "BA202", "airline": "British Airways",   "departs": "11:30", "arrives": "23:45", "price": 620, "class": "Economy"},
        {"flight": "UA303", "airline": "United Airlines",   "departs": "14:00", "arrives": "02:15", "price": 890, "class": "Business"},
    ]
    trip = f"round-trip (return: {return_date})" if return_date else "one-way"
    lines = [f"Flights from {origin} to {destination} on {departure_date} ({trip}):\n"]
    for f in flights:
        lines.append(
            f"  {f['flight']} | {f['airline']} | {f['departs']}–{f['arrives']} | "
            f"${f['price']} | {f['class']}"
        )
    return "\n".join(lines)


@tool
def search_hotels(city: str, check_in: str, check_out: str, guests: int = 1) -> str:
    """Search for available hotels in a city.

    Args:
        city: Destination city
        check_in: Check-in date in YYYY-MM-DD format
        check_out: Check-out date in YYYY-MM-DD format
        guests: Number of guests (default 1)
    """
    hotels = [
        {"name": "Grand Plaza Hotel",    "stars": 5, "price_per_night": 320, "amenities": "Pool, Spa, Restaurant"},
        {"name": "City Center Inn",       "stars": 3, "price_per_night": 95,  "amenities": "Free WiFi, Breakfast"},
        {"name": "Boutique Stay",         "stars": 4, "price_per_night": 180, "amenities": "Gym, Bar, Concierge"},
    ]
    nights = (date.fromisoformat(check_out) - date.fromisoformat(check_in)).days
    lines = [f"Hotels in {city} | {check_in} → {check_out} ({nights} nights, {guests} guest(s)):\n"]
    for h in hotels:
        total = h["price_per_night"] * nights
        lines.append(
            f"  {'★' * h['stars']} {h['name']} | ${h['price_per_night']}/night (total ${total}) | {h['amenities']}"
        )
    return "\n".join(lines)


@tool
def book_flight(flight_number: str, passenger_name: str, origin: str, destination: str, travel_date: str) -> str:
    """Book a flight for a passenger.

    Args:
        flight_number: Flight number to book (e.g. 'AA101')
        passenger_name: Full name of the passenger
        origin: Departure city or airport
        destination: Arrival city or airport
        travel_date: Travel date in YYYY-MM-DD format
    """
    global booking_counter
    booking_counter += 1
    ref = f"FLT-{booking_counter:04d}"
    bookings[ref] = {
        "type": "flight", "flight": flight_number, "passenger": passenger_name,
        "route": f"{origin} → {destination}", "date": travel_date, "status": "Confirmed"
    }
    return f"✅ Flight booked! Ref: {ref} | {flight_number} | {origin} → {destination} on {travel_date} | Passenger: {passenger_name}"


@tool
def book_hotel(hotel_name: str, guest_name: str, city: str, check_in: str, check_out: str) -> str:
    """Book a hotel room for a guest.

    Args:
        hotel_name: Name of the hotel to book
        guest_name: Full name of the guest
        city: City where the hotel is located
        check_in: Check-in date in YYYY-MM-DD format
        check_out: Check-out date in YYYY-MM-DD format
    """
    global booking_counter
    booking_counter += 1
    ref = f"HTL-{booking_counter:04d}"
    bookings[ref] = {
        "type": "hotel", "hotel": hotel_name, "guest": guest_name,
        "city": city, "check_in": check_in, "check_out": check_out, "status": "Confirmed"
    }
    return f"✅ Hotel booked! Ref: {ref} | {hotel_name} in {city} | {check_in} → {check_out} | Guest: {guest_name}"


@tool
def get_bookings() -> str:
    """Retrieve all current bookings."""
    if not bookings:
        return "No bookings found."
    lines = ["Current bookings:\n"]
    for ref, b in bookings.items():
        if b["type"] == "flight":
            lines.append(f"  [{ref}] ✈ {b['flight']} | {b['route']} on {b['date']} | {b['passenger']} | {b['status']}")
        else:
            lines.append(f"  [{ref}] 🏨 {b['hotel']} in {b['city']} | {b['check_in']} → {b['check_out']} | {b['guest']} | {b['status']}")
    return "\n".join(lines)


@tool
def cancel_booking(booking_ref: str) -> str:
    """Cancel an existing booking.

    Args:
        booking_ref: Booking reference number (e.g. 'FLT-0001' or 'HTL-0002')
    """
    if booking_ref not in bookings:
        return f"Booking {booking_ref} not found."
    bookings[booking_ref]["status"] = "Cancelled"
    return f"❌ Booking {booking_ref} has been cancelled."

# All tools return formatted strings so the agent can relay results to the user.

Now we initialize the Strands agent with a travel-assistant system prompt and all six tools defined above.

In [ ]:
# Initialize the travel-assistant agent with a system prompt and all booking tools
agent = Agent(
    system_prompt=(
        "You are a helpful travel booking assistant. You help users search for flights and hotels, "
        "make bookings, view existing reservations, and cancel bookings. "
        "Always confirm details with the user before completing a booking. "
        "Use today's date as context when dates are not fully specified."
    ),
    tools=[search_flights, search_hotels, book_flight, book_hotel, get_bookings, cancel_booking],
    callback_handler=None,  # suppress streaming output to avoid Rich/Jupyter recursion
)

## Quick Demo - Agent Invocation

Before diving into evaluation, let's verify the agent works with a single-turn request that searches for flights and books the cheapest option.

In [ ]:
# Quick smoke test: search flights and book the cheapest in a single turn
agent("Search for flights from New York to London on 2025-09-01, then book the cheapest one for Jane Doe")

# DeepEval Multi-Turn Evaluation Framework

Every change in your GenAI application, whether it is a prompt revision, a model swap, or a retrieval pipeline update, carries the risk of improving one conversational behavior while quietly degrading another. Multi-turn evaluation gives you a controlled way to measure that tradeoff. By running successive versions of your application through the same set of scenarios, you can quantify exactly where a new release improves, where it regresses, and where it holds steady, before any of those changes reach production. DeepEval's multi-turn evaluation framework helps to implement this mechanism.

DeepEval's multi-turn evaluation framework is built around three components that comprise the evaluation flow: *ConversationalGolden, ConversationSimulator*, and *ConversationalTestCase*. ConversationalGolden defines what to evaluate, ConversationSimulator simulates the conversation and applies the evaluation scenarios to your GenAI application, ConversationalTestCase captures the results of these runs for evaluation scoring.

The simulation of scenarios follows a three-stage loop that mirrors how a real user would interact with your application:

1. **Scenario initialization**. A ConversationalGolden supplies the ConversationSimulator with everything it needs to begin: the scenario description, the simulated user's persona, and the expected outcome that signals a successful resolution.
2. **Turn-by-turn generation**. The simulator produces a realistic user message based on the persona and conversation history so far, then passes it to your GenAI application, which responds as it would in production. This back-and-forth continues iteratively until either the conversation reaches the defined expected outcome or the maximum turn limit is hit.
3. **Packaging for evaluation**. Once the conversation concludes, the entire exchange, every user message and assistant reply in sequence, is bundled into a ConversationalTestCase, ready to be scored against your multi-turn metrics.

## Build Golden Scenarios with ConversationalGolden

Rather than defining a fixed script of messages and expected replies, a *ConversationalGolden* captures the conditions under which a conversation should take place. It includes a natural-language scenario description (e.g., "A budget-conscious couple planning a weekend getaway to Paris"), a persona that defines how the simulated user should behave, and an expected outcome that describes what a successful resolution looks like. 

Think of a *ConversationalGolden* as the test specification: it tells the system what to test without prescribing how the conversation will unfold.

*ConversationalGolden* fields are:

| Field | Purpose |
|---|---|
| `scenario` | Describes the conversational situation to simulate |
| `expected_outcome` | Defines what a successful resolution looks like |
| `user_description` | Sets the persona and behavioral traits of the simulated user |

*ConversationGolden* items comprise the *EvaluationDataset* that can be treated as a code, versioned, and stored in a source code control tool.

The next snippet of code defines two `ConversationalGolden` scenarios for two different customer personas. 

In [ ]:
from deepeval.dataset import ConversationalGolden

# Define two evaluation scenarios with distinct user personas
goldens=[
    ConversationalGolden(
        scenario="Solo traveler searching for a cheap one-way flight from New York to London and booking the most affordable option",
        expected_outcome="User finds a suitable flight and receives a confirmed booking reference",
        user_description="Budget-conscious traveler who prioritizes low cost over comfort or schedule"
    ),    
    ConversationalGolden(
        scenario="User who already has multiple bookings and wants to review them before deciding which one to cancel",
        expected_outcome="User retrieves all current bookings, selects one to cancel, and verifies the remaining bookings are still intact",
        user_description="Cautious user who double-checks details and asks for confirmation before making any changes"
    )    
]

print(f"Defined {len(goldens)} conversation scenarios.")

The quality of your evaluation is directly determined by the quality of the scenarios driving it. 

Target at least 30 scenarios, spread deliberately across core workflows that cover everyday usage, edge cases that probe the boundaries of your agent's capabilities, and adversarial situations where conversations are most likely to break down. 

Vary user behavior by pairing identical scenarios with distinct personas to test adaptability.

In [ ]:
# Same scenario paired with two contrasting personas to test adaptability

# Relaxed, flexible traveler
goldens.append(ConversationalGolden(
    scenario="Traveler searching for a round-trip flight from Boston to Rome",
    expected_outcome="User compares options and books a flight",
    user_description="Easygoing traveler with flexible dates who is happy to take recommendations"
))

# Demanding, detail-oriented traveler
goldens.append(ConversationalGolden(
    scenario="Traveler searching for a round-trip flight from Boston to Rome",
    expected_outcome="User compares options and books a flight",
    user_description="Meticulous planner who asks about every fare detail, baggage policy, and layover duration before committing"
))


Target edge cases explicitly rather than relying on organic variation to surface boundary conditions, and test multi-topic conversations in which users switch between tasks mid-session, as these are where context retention failures are most likely to emerge.

In [ ]:
# Edge case: invalid (past) date handling
goldens.append(ConversationalGolden(
    scenario="User tries to book a flight for a date that has already passed",
    expected_outcome="Agent recognizes the invalid date and prompts the user to provide a future date",
    user_description="Distracted user who does not realize they entered the wrong date"
))

# Multi-topic conversation: flight + hotel booking then partial cancellation
goldens.append(ConversationalGolden(
    scenario="User searches for flights to Tokyo, then pivots to finding a hotel, books both, and finally asks to cancel just the hotel",
    expected_outcome="User ends with a confirmed flight booking and a successfully canceled hotel booking",
    user_description="Decisive but unpredictable user whose plans shift as the conversation progresses"
))


## 2.2 Simulate User Conversation with ConversationSimulator

*ConversationSimulator* reads the scenario and persona from a *ConversationalGolden*, generates a realistic opening user message, sends it to your application via the *model_callback* (see below), receives the assistant's reply, and then generates the next user message based on everything said so far. This back-and-forth loop continues until the conversation either reaches the expected outcome defined in the golden or hits a configured turn limit. The result is a completely generated conversation that is different every time it runs, yet always grounded in the same scenario constraints.

By default, DeepEval uses OpenAI models for user conversation simulation. To use Bedrock instead, the following code creates the corresponding custom DeepEval model based on Amazon Nova Micro for efficiency.

In [ ]:
import json
import boto3
from deepeval.models import DeepEvalBaseLLM


class BedrockNovaMicro(DeepEvalBaseLLM):
    """DeepEval-compatible wrapper around Amazon Nova Micro via Bedrock.
    Used as the simulator and evaluation judge model instead of the default OpenAI backend."""

    MODEL_ID = "amazon.nova-micro-v1:0"

    def __init__(self, region: str = "us-east-1"):
        self.region = region
        super().__init__(model=self.MODEL_ID)

    def load_model(self):
        return boto3.client("bedrock-runtime", region_name=self.region)

    def get_model_name(self) -> str:
        return self.MODEL_ID

    @staticmethod
    def _flatten_values(obj):
        """Recursively convert non-string values in dicts to JSON strings."""
        if isinstance(obj, dict):
            for k, v in obj.items():
                if isinstance(v, dict):
                    obj[k] = json.dumps(v)
                elif isinstance(v, list):
                    obj[k] = [json.dumps(i) if not isinstance(i, str) else i for i in v]
        elif isinstance(obj, list):
            for item in obj:
                if isinstance(item, dict):
                    BedrockNovaMicro._flatten_values(item)

    def _invoke(self, prompt: str, schema=None) -> str:
        system_text = "You are a helpful assistant. Respond only in valid JSON." if schema else None
        if schema:
            json_schema = schema.model_json_schema()
            prompt = (
                f"{prompt}\n\n"
                f"Return ONLY valid JSON matching this schema (no markdown, no extra text):\n"
                f"{json.dumps(json_schema, indent=2)}"
            )

        messages = [{"role": "user", "content": [{"text": prompt}]}]
        body = {"messages": messages, "inferenceConfig": {"maxTokens": 4096, "temperature": 0.0}}
        if system_text:
            body["system"] = [{"text": system_text}]

        response = self.model.converse(modelId=self.MODEL_ID, **body)
        text = response["output"]["message"]["content"][0]["text"]

        if schema:
            try:
                return schema.model_validate_json(text)
            except Exception:
                data = json.loads(text)
                self._flatten_values(data)
                return schema.model_validate(data)
        return text

    def generate(self, prompt: str, schema=None) -> str:
        return self._invoke(prompt, schema)

    async def a_generate(self, prompt: str, schema=None) -> str:
        return self._invoke(prompt, schema)


# Instantiate the Bedrock model for use as simulator and evaluation judge
nova_micro = BedrockNovaMicro()
print(f"Model loaded: {nova_micro.get_model_name()}")

A response callback, `agent_callback`, is used by simulator for producing application responses. It wraps your conversational assistant and returns valid Turns from the obtained responses.

In [ ]:
from deepeval.test_case import ToolCall, Turn


def agent_callback(input: str) -> Turn:
    """Invoke the Strands agent and return a DeepEval Turn. 
       We use a local agent running in the notebook, however, in a real-world scenario you 
       would typically invoke a remote agent API for agent deplyed to production runtime, 
       such as the one provided by AgentCore runtime running the agent service."""

    response = agent(input)
    text = response.message["content"][0]["text"] if response.message.get("content") else str(response)
    tools = [ToolCall(name=tm.tool["name"], 
                      input_parameters=tm.tool.get("input")) 
             for tm in response.metrics.tool_metrics.values()] or None
    return Turn(role="assistant", 
                content=text,
                tools_called=tools)

Next, we create the `ConversationSimulator` instance, wiring it to our agent callback and the Nova Micro model for user message generation.

In [ ]:
from deepeval.simulator import ConversationSimulator

# Create the simulator: generates user messages and feeds them to agent_callback
simulator = ConversationSimulator(model_callback=agent_callback,                                   
                                  simulator_model=nova_micro,
                                  async_mode=False)

The simulator uses Amazon Nova Micro model for user simulation generation and cannot exceed 10 turns during the simulation. For production workloads, increase this number to 20-30. 
Now let's start the simulation.

In [ ]:
# Run the simulation: generate synthetic conversations for each golden scenario
n_goldens = len(goldens)
test_cases = simulator.simulate(
    conversational_goldens=goldens,
    max_user_simulations=10,  # max turns per conversation
    on_simulation_complete=lambda tc, i: print(f"Finished test case #{i+1} out of {n_goldens} test cases"),
)

*ConversationalTestCase* is the output of that simulation and the input to evaluation. It packages the full conversation, every user message and assistant response in order, into a structured object that DeepEval's multi-turn metrics can score. It preserves the sequential relationship between turns, so metrics can assess not just individual reply quality but also consistency, context retention, and conversation flow across the entire session.

The following code prints a preview of each simulated conversation, showing the role and a truncated content snippet for every turn.

In [ ]:
# Display a summary of each simulated conversation
print(f"\nSimulated {len(test_cases)} conversations.")
for i, conv in enumerate(test_cases):
    print(f"\n--- Conversation {i+1} ({len(conv.turns)} turns) ---")
    for turn in conv.turns:
        preview = turn.content[:120].replace("\n", " ")
        print(f"  [{turn.role}] {preview}..." if len(turn.content) > 120 else f"  [{turn.role}] {preview}")

With our test cases generated, we will score the conversations against a suite of multi-turn evaluation metrics designed to assess different dimensions of conversational quality.

## Evaluation Metrics

DeepEval organizes multi-turn metrics into different types, each designed to detect a distinct class of conversational failure. Applied together, they provide comprehensive coverage of the conversational behaviors most likely to degrade in a multi-turn GenAI pipeline.

| Category | What It Evaluates | Key Metrics |
|---|---|---|
| Conversation Quality | Overall success, turn-level relevance, <br>and context retention | `ConversationCompletenessMetric`, <br>`TurnRelevancyMetric`, <br>`KnowledgeRetentionMetric` |
| Behavioral Compliance | Adherence to assigned roles <br>and topic boundaries | `RoleAdherenceMetric`, <br>`TopicAdherenceMetric` |
| Agentic | Goal completion accuracy <br>and correctness of tool invocations | `GoalAccuracyMetric`, <br>`ToolUseMetric` |
| Custom | Any evaluation criteria <br>defined by the developer | `ConversationalGEval`, <br>`ConversationalDAGMetric` |

### Conversation Quality Metrics

Conversation quality metrics assess whether the dialogue fulfills its intended objective, whether each response is contextually appropriate within the broader exchange, and whether the assistant maintains accurate recall of information introduced in earlier turns.

| Metric | What It Measures | When to Use | Score |
|---|---|---|---|
| `ConversationCompletenessMetric` | Rate of user intentions <br>satisfied by the assistant. <br>Extracts intentions from user turns, <br>checks if each was satisfied. | Always. The most important <br>multi-turn metric: did the <br>conversation succeed? | `Satisfied Intentions` <br>/ `Total Intentions` |
| `TurnRelevancyMetric` | Response relevance to <br>preceding conversational context. <br>Uses a sliding window approach. | Always. Catches non-sequiturs, <br>context overflow, and ignored <br>user messages. | `Relevant Turns` <br>/ `Total Assistant Turns` |
| `KnowledgeRetentionMetric` | Retention of facts the user <br>shared earlier (names, preferences, <br>requirements). | Information-rich conversations <br>(onboarding, travel planning, <br>multi-step booking). | `Turns Without Attrition` <br>/ `Total Assistant Turns` |


To collect these metrics, pass them together with the collected simulated conversations to evaluate().

Before we start evaluating, basic evaluation components should be imported.

In [ ]:
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig

Let's create the evaluation with three major metrics - `ConversationCompletenessMetric`, `TurnRelevancyMetric` and `KnowledgeRetentionMetric`.

In [ ]:
from deepeval.test_case import (
    Turn, 
    ConversationalTestCase
)

from deepeval.metrics import (
    ConversationCompletenessMetric,
    TurnRelevancyMetric,
    KnowledgeRetentionMetric,
)

# Score all simulated conversations against conversation quality metrics
evaluate(    
    test_cases=test_cases,
    metrics=[
        ConversationCompletenessMetric(model=nova_micro),
        TurnRelevancyMetric(model=nova_micro),
        KnowledgeRetentionMetric(model=nova_micro)
    ],
    async_config=AsyncConfig(run_async=False),
)


### Behavioral Compliance Metrics

These metrics ensure the assistant stays within its designated boundaries, both in terms of persona and of topic scope. 

| Metric | What It Measures | When to Use | Score |
|---|---|---|---|
| `RoleAdherenceMetric` | Whether the assistant stays <br>in character throughout <br>the conversation. Requires <br>`chatbot_role` on the test case. | Application has a defined <br>persona, behavioral guidelines, <br>or scope restrictions. | `Turns Adhering to Role` <br>/ `Total Assistant Turns` |
| `TopicAdherenceMetric` | Whether the assistant engages <br>only with on-topic questions <br>and refuses off-topic ones. <br>Classifies QA pairs against <br>`relevant_topics`. | Application should only <br>respond to specific <br>subject areas. | `(TP + TN)` <br>/ `Total QA Pairs` |


To collect these metrics define the corresponding evaluation varilable - `chatbot_role` and `relevant_topics`.

In [ ]:
from deepeval.metrics import (
    RoleAdherenceMetric,
    TopicAdherenceMetric
)

# Define the expected chatbot role for behavioral compliance evaluation
CHATBOT_ROLE = "A travel booking assistant that helps users search flights and hotels, place reservations, and manage or cancel bookings."

for tc in test_cases:
    tc.chatbot_role = CHATBOT_ROLE

print(f"Set chatbot_role on {len(test_cases)} test case(s).")

# Evaluate role adherence and topic adherence
adherence_metrics = [
RoleAdherenceMetric(    
    threshold=0.7,
    model=nova_micro
),
TopicAdherenceMetric(
    relevant_topics=["flight search", "hotel search", "booking management"],
    threshold=0.7,
    model=nova_micro
)
]

evaluate(    
    test_cases=test_cases,
    metrics=adherence_metrics,
    async_config=AsyncConfig(run_async=False)
)

### Agentic Multi-Turn Metrics

These metrics evaluate goal-oriented behavior and tool-using within multi-turn conversations.

| Metric | What It Measures | When to Use | Score |
|---|---|---|---|
| `GoalAccuracyMetric` | Whether the assistant can plan <br>and execute tasks to reach <br>a goal. Assesses both plan <br>quality and execution accuracy. | Task-completion apps <br>(booking systems, workflow <br>assistants, multi-step agents). | `(Goal Eval Score` <br>`+ Plan Eval Score) / 2` |
| `ToolUseMetric` | Correct tool selection and <br>argument generation across turns. <br>Requires `available_tools` <br>on the metric. | Application uses tools <br>or function calls. Catches <br>wrong tools and bad arguments. | `min(Tool Selection,` <br>`Argument Correctness)` |

The code below registers all six agent tools with `ToolUseMetric` and evaluates both metrics against the simulated conversations.

In [ ]:
from deepeval.metrics import (
    ToolUseMetric,
    GoalAccuracyMetric
)

# Define available tools so ToolUseMetric can verify correct tool selection
metrics = [
    ToolUseMetric(
        available_tools=[
            ToolCall(name="search_flights", description="Search for available flights between two cities"),
            ToolCall(name="search_hotels", description="Search for available hotels in a city"),
            ToolCall(name="book_flight", description="Book a flight for a passenger"),
            ToolCall(name="book_hotel", description="Book a hotel room for a guest"),
            ToolCall(name="get_bookings", description="Retrieve all current bookings"),
            ToolCall(name="cancel_booking", description="Cancel an existing booking"),
        ],
        model=nova_micro,
        threshold=0.6,
    ),
    GoalAccuracyMetric(threshold=0.6, model=nova_micro),
]

evaluate(test_cases=test_cases, metrics=metrics, async_config=AsyncConfig(run_async=False))


### Custom Multi-Turn Metrics

The built-in metrics address the most common categories of conversational failure, but every application carries domain-specific requirements that generic metrics cannot fully capture. For these cases, DeepEval provides the ability to define custom multi-turn metrics tailored to your evaluation needs.

**ConversationalGEval** expresses evaluation criteria in natural language and lets an LLM judge score the conversation accordingly.
**ConversationalDAGMetric** builds a deterministic decision tree (DAG) for structured, multi-step evaluation logic. Rather than expressing criteria as a single natural-language string, you define a multi-step decision tree that the metric traverses node by node to arrive at a final score.

| Metric | What It Measures | When to Use | How It Is Calculated |
|---|---|---|---|
| `ConversationalGEval` | Whether the conversation meets <br>custom criteria defined in <br>natural language (tone, empathy, <br>brand voice, policy compliance). | Evaluating subjective qualities <br>not covered by built-in metrics. | Generates eval steps via <br>chain-of-thought, applies them <br>across the conversation, <br>normalizes with LLM token probs. |
| `ConversationalDAGMetric` | Whether the conversation <br>satisfies a deterministic <br>decision tree (DAG). | Branching evaluation logic <br>(e.g., check goal first, <br>then assess tone only if met). | Traverses DAG in topological <br>order; LLM judge at each node <br>decides the branch, arriving <br>at a verdict with final score. |

The code below demonstrates both approaches:

1. **Empathy metric (ConversationalGEval)** — We define a single natural-language criterion: *"Does the assistant show genuine empathy when the user expresses frustration?"* DeepEval automatically decomposes this into chain-of-thought evaluation steps and applies them across the full conversation. The threshold is set to 0.7, meaning a conversation passes only if the empathy score reaches at least 70%.

2. **Behavior metric (ConversationalDAGMetric)** — We build a three-level decision tree:
   - **TaskNode** (root): Summarizes the entire conversation and the assistant's overall behavior.
   - **BinaryJudgementNode**: Asks *"Did the assistant satisfy the user's questions?"* If no → score 0. If yes → proceed to the next node.
   - **NonBinaryJudgementNode**: Classifies the assistant's tone as Rude (score 0), Playful (score 5), or Neutral (score 10).

   This branching structure ensures tone is only evaluated when the assistant actually addressed the user's needs — there is no point scoring politeness if the core task failed.

Note: The `make_behavior_metric()` factory function creates a fresh DAG instance for each evaluation run. This is a workaround for a DeepEval issue where internal `turn_window` state can carry over between test cases, causing incorrect scoring. The monkey-patch on `is_valid_turn_window` serves the same purpose.

In [ ]:
from deepeval.test_case import TurnParams
from deepeval.metrics import ConversationalGEval
from deepeval.metrics import ConversationalDAGMetric
from deepeval.metrics.dag import DeepAcyclicGraph
from deepeval.metrics.conversational_dag import (
    ConversationalTaskNode,
    ConversationalBinaryJudgementNode,
    ConversationalNonBinaryJudgementNode,
    ConversationalVerdictNode,
)
from deepeval.evaluate import AsyncConfig

# ConversationalGEval: LLM-judged empathy scoring using natural-language criteria
empathy_metric = ConversationalGEval(
    name="Empathy",
    criteria="Evaluate whether the assistant shows genuine empathy when the user expresses frustration or dissatisfaction.",
    model=nova_micro,
    threshold=0.7,
)


def make_behavior_metric():
    """Create a fresh ConversationalDAGMetric (DAG) to avoid stale turn_window state.
    The DAG first summarizes the conversation, checks if user goals were met,
    and only then branches into a tone assessment (Rude / Playful / Neutral)."""
    non_binary_node = ConversationalNonBinaryJudgementNode(
        criteria="How was the assistant's behaviour towards the user?",
        children=[
            ConversationalVerdictNode(verdict="Rude", score=0),
            ConversationalVerdictNode(verdict="Playful", score=5),
            ConversationalVerdictNode(verdict="Neutral", score=10),
        ],
    )

    binary_node = ConversationalBinaryJudgementNode(
        criteria="Do the assistant's replies satisfy the user's questions?",
        children=[
            ConversationalVerdictNode(verdict=False, score=0),
            ConversationalVerdictNode(verdict=True, child=non_binary_node),
        ],
    )

    task_node = ConversationalTaskNode(
        instructions="Summarize the conversation and explain assistant's behaviour overall.",
        output_label="Summary",
        evaluation_params=[TurnParams.ROLE, TurnParams.CONTENT],
        children=[binary_node],
    )

    dag = DeepAcyclicGraph(root_nodes=[task_node])
    return ConversationalDAGMetric(name="Behavior", model=nova_micro, dag=dag)


# Workaround: patch DAG nodes to reset turn_window before each test case,
# preventing stale indices from a longer conversation crashing on a shorter one.
import deepeval.metrics.conversational_dag.nodes as _dag_nodes

_orig_task_execute = _dag_nodes.ConversationalTaskNode._execute
_orig_binary_execute = _dag_nodes.ConversationalBinaryJudgementNode._execute
_orig_nonbinary_execute = _dag_nodes.ConversationalNonBinaryJudgementNode._execute

def _patched_execute(orig_fn):
    def wrapper(self, *args, **kwargs):
        self.turn_window = None  # force recalculation for this test case
        return orig_fn(self, *args, **kwargs)
    return wrapper

_dag_nodes.ConversationalTaskNode._execute = _patched_execute(_orig_task_execute)
_dag_nodes.ConversationalBinaryJudgementNode._execute = _patched_execute(_orig_binary_execute)
_dag_nodes.ConversationalNonBinaryJudgementNode._execute = _patched_execute(_orig_nonbinary_execute)

# Run custom metrics (empathy + behavior DAG) on all test cases
evaluate(
    test_cases=test_cases,
    metrics=[empathy_metric, make_behavior_metric()],
    async_config=AsyncConfig(run_async=False),
)

# Restore original _execute methods
_dag_nodes.ConversationalTaskNode._execute = _orig_task_execute
_dag_nodes.ConversationalBinaryJudgementNode._execute = _orig_binary_execute
_dag_nodes.ConversationalNonBinaryJudgementNode._execute = _orig_nonbinary_execute


### Interpreting the Results

The output above shows evaluation results for both custom metrics across all test cases.

**Empathy metric** — Each test case receives a score between 0.0 and 1.0. A score above the 0.7 threshold means the assistant demonstrated adequate empathy during the conversation. Low scores indicate turns where the user expressed frustration or dissatisfaction and the assistant responded in a flat or dismissive tone. Since our travel assistant scenarios are mostly transactional, empathy scores may be high by default; this metric becomes more revealing when you add adversarial scenarios where the user is upset (e.g., a canceled flight or a wrong booking).

**Behavior metric (DAG)** — The score reflects the path taken through the decision tree:
- **Score 0**: The assistant failed to satisfy the user's questions (binary node returned False), so tone was never evaluated.
- **Score 5**: The assistant satisfied the user's questions and its tone was classified as Playful.
- **Score 10**: The assistant satisfied the user's questions and its tone was classified as Neutral (the ideal for a professional travel assistant).

If you see a score of 0 on a scenario that should have succeeded, inspect the conversation transcript — the issue is likely a task-completion failure rather than a tone problem. If scores cluster around 5 instead of 10, the assistant may be using overly casual language for a professional context.

Together, these two custom metrics demonstrate the flexibility of DeepEval's custom evaluation framework: `ConversationalGEval` for quick, natural-language criteria that an LLM judge can assess holistically, and `ConversationalDAGMetric` for structured, multi-step evaluation logic where the scoring path depends on earlier judgments.

## Best Practices for Evaluation Metrics

| Practice | Guidance |
|----------|----------|
| Start with at least 30 scenarios | Cover core workflows, edge cases, <br>and adversarial situations. Pair the same <br>scenario with different personas <br>to test adaptability. |
| Run evaluations on every change | Treat your scenario suite like a test suite: <br>run it after every prompt revision, model swap, <br>or pipeline update to catch regressions early. |
| Increase turn limits for production | The 10-turn cap used here is for demonstration. <br>Real conversations often need 20–30 turns <br>to surface context retention <br>and coherence failures. |
| Combine metric categories | No single metric tells the full story. <br>Use quality, behavioral, and agentic metrics <br>together for comprehensive coverage. |
| Use custom metrics for domain needs | Built-in metrics cover common failure modes. <br>Brand voice, regulatory compliance, and tone <br>require `ConversationalGEval` <br>or `ConversationalDAGMetric`. |
| Version your golden scenarios | Store `ConversationalGolden` definitions <br>in source control alongside your application code <br>so evaluation criteria evolve with the product. |
| Inspect low scores before tuning | A low score may indicate a scenario design issue <br>rather than an application defect. <br>Always review the conversation transcript first. |


# Next Steps

- Continue to **`04-11-05-strands-evaluators.ipynb`** to learn how to use Strands Eval to evaluate multi-turn conversations
- Or skip to **`04-11-06-synthetic-data.ipynb`** to learn how to generate diverse test scenarios automatically